# AIGC 5005 · Module 05 · Part 4
## Cross-validation

Companion notebook for the matching part of the Module 05 learning content. Read each explanation, **predict** what the next cell will show, run it, and compare. Change values and run again; you cannot break anything.

**Kernel:** choose the `module05-sklearn` conda environment (top right in VS Code). To make it: `conda create -n module05-sklearn python=3.11 ipykernel`, then `conda activate module05-sklearn` and `pip install -r requirements.txt`. `churn_pipeline.py` and `customers.py` in this folder are shared modules.

In [1]:
from churn_pipeline import build_pipeline, load_split

X, y, X_train, X_test, y_train, y_test = load_split()
pipe = build_pipeline()

## One split, one number
The score depends on which rows landed in the test set.

In [2]:
from sklearn.model_selection import train_test_split

for seed in (1, 2, 3):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
    print(seed, round(pipe.fit(Xtr, ytr).score(Xte, yte), 3))

1 0.864
2 0.862
3 0.858


## Five-fold cross-validation
Report the **mean** as the estimate and the **standard deviation** as how much to trust it.

In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X, y, cv=folds, scoring="accuracy")
print(scores.round(3), scores.mean().round(3), scores.std().round(3))

[0.862 0.86  0.86  0.858 0.862] 0.86 0.002


## More than one metric

In [4]:
import pandas as pd
from sklearn.model_selection import cross_validate

results = cross_validate(pipe, X, y, cv=folds, scoring=["accuracy", "precision", "recall", "roc_auc"])
table = pd.DataFrame({name: results["test_" + name] for name in ["accuracy", "precision", "recall", "roc_auc"]})
print(table.round(3))
for name in table:
    print(f"{name:10s} {table[name].mean():.3f}")

   accuracy  precision  recall  roc_auc
0     0.862      0.545   0.107    0.705
1     0.860      0.500   0.089    0.792
2     0.860      0.500   0.071    0.735
3     0.858      0.500   0.035    0.758
4     0.862      0.750   0.053    0.805
accuracy   0.861
precision  0.559
recall     0.071
roc_auc    0.759


Accuracy looks fine, but recall is only about 0.07. Part 5 explains why.

## Try it
1. Change `n_splits` to 10. Which changes more, the mean or the spread?
2. Remove `shuffle=True` and `random_state`, run twice, and notice the folds are no longer repeatable.